# Picoからのシリアル通信を受け取る
- 加速度センサ

In [1]:
import serial
import array

from IPython.display import clear_output

In [2]:
import numpy as np
import datetime
import time
# Poseの各ランドマークを保存
class body1():
    def __init__(self, dirName="motion"):
        self.dirName = dirName
        self.fileName = "motion"
        self.vec_size = 1
        
        # 以下は触らない
        self.isRecord = False
        self.data = np.empty((0,self.vec_size,3), dtype=np.float32)
        self.time_ary = []
        
        
    def record1(self, values):
        if self.isRecord == False:
            return False
        
        data = np.empty((self.vec_size, 3), dtype=np.float32)
        for i in range(self.vec_size):
            data[i] = LandmarkToArray(values[i])

        self.data = np.vstack([self.data, np.array([data])])
        self.time_ary.append(time.perf_counter())
        return True

    
    def save(self, data_label):
        # numpy行列を保存
        data = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        filename_data = self.dirName + "/" + self.fileName + data
        #np.save(filename_data, self.data)
        print(filename_data)
        print(self.data.shape)
        print(self.data)
        # CSVに保存
        print(len(self.time_ary))
        for i in range(len(self.time_ary) - 1):
            self.time_ary[i+1] = self.time_ary[i+1] - self.time_ary[0]
        self.time_ary[0] = 0.0
        frame_ary = list(range(len(self.time_ary)))
        label1 = np.array(data_label * 3)
        label1 = label1.reshape([3, self.vec_size]).reshape(-1, order='F')
        label2 = ['X', 'Y', 'Z'] * self.vec_size
        label = "Frame,Time," + ",".join(label1) + "\n" + ",," + ",".join(label2)
        csv_data = self.data.reshape([self.data.shape[0], 3*self.vec_size])
        csv_data = np.insert(csv_data, 0, self.time_ary, axis=1)
        csv_data = np.insert(csv_data, 0, frame_ary, axis=1)
        np.savetxt(filename_data + ".csv", csv_data, delimiter=',', 
                   header=label, comments='',
                   fmt='%f')
        
    def start_stop(self):
        if self.isRecord == False:
            self.isRecord = True
            clear_output(wait=True)
            print("Start")
            return
        else:
            label = ["acc1"]
            self.save(label)
            print("Recorded")
            self.data = np.empty((0,self.vec_size,3), dtype=np.float32)
            self.time_ary = []
            self.isRecord = False
            return

In [ ]:
ser = serial.Serial("COM3", 9600)
learnData = body1("test1")

def read_acceralation():
    while (ser.read(1) == 0xFF):
        pass
    ax = ser.read(1)
    ay = ser.read(1)
    az = ser.read(1)
    ax = int.from_bytes(ax, 'big')
    ay = int.from_bytes(ay, 'big')
    az = int.from_bytes(az, 'big')
    return [ax, ay, az]

while (True):
    acc = read_acceralation()
    
    clear_output(wait=True)
    print(acc[0])
    print(acc[1])
    print(acc[2])
    
    #result_sagittal = mediapipe2.loop()
    key = cv2.waitKey(1) & 0xFF
    if key == 27:    # Escで終了
        break;
    if key == ord(' '):    # Spaceキーで録画・終了
        learnData.start_stop()

    learnData.record1(acc)
    
ser.close()

In [39]:
ser.close()